### ⋆✴︎˚｡⋆ An AI model that when presented with gene expression data from skin samples, detects biomarkers for psoriasis and outputs a probability/risk score for psoriasis.  
# ------------------------------------------------------
#### Dataset used: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE13355  
#### **GSE13355 — 122 skin samples used out of 180 total (58 PP, 58 PN, 64 NN)**
#### -PP (psoriasis) 
#### -NN (healthy) 
#### *not including PN (healthy skin from psoriasis patients) for simplicity*
#### 54,675 genes reduced to 10,923 after cleaning

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import GEOparse

print("All libraries loaded! Yay!")

All libraries loaded! Yay!


In [3]:
# gse = GEOparse.get_GEO(geo="GSE13355", destdir="./data")
# print("Download complete!")  
# ^^ used this code to get the dataframe from the ncbi GEO. Since it's already in my project files... 
df = pd.read_csv("data/cleaned_expression.csv", index_col=0)
y = pd.read_csv("data/labels.csv", index_col=0).squeeze()

print("Data loaded!")
print("Shape:", df.shape)
print("Psoriasis (1):", sum(y == 1))
print("Healthy (0):", sum(y == 0))

Data loaded!
Shape: (122, 10923)
Psoriasis (1): 58
Healthy (0): 64


In [4]:
print("Number of samples:", len(gse.gsms))
print("Number of platforms:", len(gse.gpls))
print()
print("First 5 sample names:")
for name in list(gse.gsms.keys())[:5]:
    print(" ", name)

NameError: name 'gse' is not defined

In [ ]:
first_sample = gse.gsms["GSM337197"]

print("Sample title:", first_sample.metadata["title"][0])
print("Sample type:", first_sample.metadata["source_name_ch1"][0])
# print("Gene expression data (first 5 rows):")
print(first_sample.table.head())

In [ ]:
print("genes per sample:", len(first_sample.table))

In [ ]:
for name, sample in list(gse.gsms.items())[:15]:
    title = sample.metadata["title"][0]
    print(name, "→", title)

In [ ]:
pp_count = 0  # psoriasis lesional
pn_count = 0  # psoriasis uninvolved
nn_count = 0  # healthy 

for name, sample in gse.gsms.items():
    title = sample.metadata["title"][0]
    if "PP" in title:
        pp_count += 1
    elif "PN" in title:
        pn_count += 1
    elif "NN" in title:
        nn_count += 1
#how many in each category, pp, pn, and nn
print("PP (psoriasis lesional):", pp_count)
print("PN (psoriasis uninvolved):", pn_count)
print("NN (healthy normal):", nn_count)
print("Total:", pp_count + pn_count + nn_count) 


In [ ]:
samples = {}
labels = {}
#Get rid of PN, as it will confuse the model at this stage for being "in between"
for name, sample in gse.gsms.items():
    title = sample.metadata["title"][0]
    if "PP" in title:
        samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        labels[name] = 1  
    elif "NN" in title:
        samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        labels[name] = 0 
#the dataframe we will use
df = pd.DataFrame(samples).T
print("Shape of our data table:", df.shape)
print("(rows = different patient, columns = genes)")

In [ ]:
df.head()

In [ ]:
y = pd.Series(labels)

print("Number of labels:", len(y))
print("Psoriasis (1):", sum(y == 1))
print("Healthy (0):", sum(y == 0))
print()
print("Do labels match data rows?", list(y.index) == list(df.index))

In [ ]:
print("Genes before removing controls:", df.shape[1])

# remove any column that starts with AFFX, these are the controls for the researchers 
df = df.loc[:, ~df.columns.str.startswith("AFFX")]

# count after
print("Genes after removing controls:", df.shape[1])

In [ ]:
# count before
print("Genes before removing missing values:", df.shape[1])

# drop any column that has even one missing value
df = df.dropna(axis=1)

# count after
print("Genes after removing missing values:", df.shape[1])

In [ ]:
#removing genes of low variance. If a gene has same level across all samples 
#it is basically useless 

# calculate variance for every gene
gene_variances = df.var()

print("Lowest variance:", gene_variances.min().round(4))
print("Highest variance:", gene_variances.max().round(4))
print("Average variance:", gene_variances.mean().round(4))

In [ ]:
#visualization is key.

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(gene_variances, bins=80, color='pink', edgecolor='steelblue')
plt.xlabel("Variance")
plt.ylabel("Number of genes")
plt.title("Distribution of gene variances across all 54613 genes")
plt.axvline(x=0.1, color='red', linestyle='--', label='potential cutoff')
plt.legend()
plt.show()

In [ ]:
# calculate the 80th percentile variance cutoff (the genes that vary the least)
cutoff = gene_variances.quantile(0.80)
print("Variance cutoff:", cutoff.round(4))

# keep only genes above that cutoff
df = df.loc[:, gene_variances > cutoff]

print("Genes before:", 54613)
print("Genes after:", df.shape[1])

In [ ]:
# save the cleaned data table
df.to_csv("data/cleaned_expression.csv")

# save the labels
y.to_csv("data/labels.csv")

print("Saved!")
print("Data shape:", df.shape)

In [ ]:
df_psoriasis = df[y == 1]
df_healthy = df[y == 0]
#avg expression per group
avg_psoriasis = df_psoriasis.mean()
avg_healthy = df_healthy.mean()

plt.figure(figsize=(8, 8)) 
plt.scatter(avg_healthy, avg_psoriasis, alpha=0.1, s=1, color='pink')
plt.xlabel("Average expression in healthy skin")
plt.ylabel("Average expression in psoriasis skin")
plt.title("Gene expression: Psoriasis vs Healthy")
plt.plot([-3, 3], [-3, 3], color='steelblue', linestyle='--', label='no difference')
plt.legend()
plt.show()

In [ ]:
from scipy import stats

# t-test for every single gene
pvalues = {}
tstatistics = {}

for gene in df.columns:
    psoriasis_values = df_psoriasis[gene]
    healthy_values = df_healthy[gene]
    t_stat, p_val = stats.ttest_ind(psoriasis_values, healthy_values)
    pvalues[gene] = p_val
    tstatistics[gene] = t_stat

# convert to a new dataframe
results = pd.DataFrame({
    'pvalue': pvalues,
    'tstatistic': tstatistics
})

print("T-tests meaaawhjajksadj")
print("Genes tested:", len(results))
print("Genes with p < 0.05:", sum(results['pvalue'] < 0.05))

*bonferroni correction* is used to prevent false positives when running  
multiple tests on the same dataset 
it keeps the error rate controlled by dividing the original significance level 
by the number of tests

In [ ]:
# apply Bonferroni correction 
bonferroni_cutoff = 0.05 / len(results)
print("Bonferroni cutoff:", bonferroni_cutoff)

significant_genes = results[results['pvalue'] < bonferroni_cutoff]
print("Genes after Bonferroni correction:", len(significant_genes))

# sort by most significant (smallest p-value first)
significant_genes = significant_genes.sort_values('pvalue')
print("\nTop 10 most significant genes:")
print(significant_genes.head(10))

In [ ]:
#still way too many freaking genes oml 
#new dataframe will include only top 300  
top_genes = significant_genes.head(300).index.tolist()
df_filtered = df[top_genes]
print("Final feature set shape:", df_filtered.shape)
print("(rows = samples, columns = selected genes)")

In [ ]:
df_filtered.to_csv("data/filtered_expression.csv") 
print("saved mwahaha")

### *SUMMARY OF DATA CLEANING*
- raw genes from the chip = 54675
- removed control probes (AFFX-) = 54613
- removed low variance genes = 10923
- kept only statistically significant ones = 300

Now it's time to build the model: first by splitting the data into a training set and a test set

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df_filtered,  # our 300 genes
    y,            # our labels (0 or 1)
    test_size=0.2,        # 20% for testing
    random_state=42,      # makes the split reproducible
    stratify=y            # keeps psoriasis/healthy ratio equal in both splits
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

In [ ]:
# scale the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# train the model
model = LogisticRegression(
    max_iter=1000,       # gives it enough time to converge
    random_state=42
)
model.fit(X_train_scaled, y_train)

print("Model trained!")

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# get predictions on test set
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

# print results
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Healthy', 'Psoriasis']))
print("AUROC Score:", roc_auc_score(y_test, y_prob).round(4))
#pls work oml 
#ignore error

In [ ]:
print("AUROC Score:", round(roc_auc_score(y_test, y_prob), 4))


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
# combine scaling and model into one pipeline to prevent data leakage
#oh god pls work
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])
scores = cross_val_score(pipeline, df_filtered, y, cv=5, scoring='roc_auc')
print("AUROC per fold:", scores.round(4))
print("Average AUROC:", scores.mean().round(4))
print("Standard deviation:", scores.std().round(4))

In [ ]:
def psoriasis_risk(sample):
    #probability D:
    prob = model.predict_proba(sample)[0][1]
    

    print("Psoriasis Risk Assessment")
    print(f"Risk Score: {prob:.1%}")
    if prob >= 0.7:
        print("Risk Level: HIGH")
    elif prob >= 0.4:
        print("Risk Level: MODERATE")
    else:
        print("Risk Level: LOW")

# test it on the first test sample
psoriasis_risk(X_test_scaled[:1])
print("Actual label:", "Psoriasis" if y_test.iloc[0] == 1 else "Healthy")  

print()
print("Sample 2")
psoriasis_risk(X_test_scaled[1:2])
print("Actual:", "Psoriasis" if y_test.iloc[1] == 1 else "Healthy")

print()
print("Sample 3")
psoriasis_risk(X_test_scaled[2:3])
print("Actual:", "Psoriasis" if y_test.iloc[2] == 1 else "Healthy")

*pickle* is a library used to convert complex objects into a byte steam (pickling) that can be saved to a file/database/transmitted to a network and later reconstructed back into the original object (unpickling) 

Here, it is used to save trained models as .pkl files to avoid retraining

In [ ]:
#damn i didnt think that would actually work
import pickle
# save the model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# save the scaler
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

# save the list of top genes
with open("top_genes.pkl", "wb") as f:
    pickle.dump(top_genes, f)

print("Model saved!")
print("Scaler saved!")
print("Top genes saved!")

## *external validation*  
### making sure the model actually knows psoriasis with the GSE14905 dataset

In [10]:
gse2 = GEOparse.get_GEO(geo="GSE14905", destdir="./data")
print("Download complete!")

14-Jun-2026 19:30:57 DEBUG utils - Directory ./data already exists. Skipping.
14-Jun-2026 19:30:57 INFO GEOparse - Downloading ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz to ./data\GSE14905_family.soft.gz
100%|██████████| 47.9M/47.9M [00:03<00:00, 13.6MB/s]
14-Jun-2026 19:31:02 DEBUG downloader - Size validation passed
14-Jun-2026 19:31:02 DEBUG downloader - Moving C:\Users\artkm\AppData\Local\Temp\tmpcd3hv2nr to C:\Users\artkm\Documents\ai-psoriasis-project\data\GSE14905_family.soft.gz
14-Jun-2026 19:31:02 DEBUG downloader - Successfully downloaded ftp://ftp.ncbi.nlm.nih.gov/geo/series/GSE14nnn/GSE14905/soft/GSE14905_family.soft.gz
14-Jun-2026 19:31:02 INFO GEOparse - Parsing ./data\GSE14905_family.soft.gz: 
14-Jun-2026 19:31:02 DEBUG GEOparse - DATABASE: GeoMiame
14-Jun-2026 19:31:02 DEBUG GEOparse - SERIES: GSE14905
14-Jun-2026 19:31:02 DEBUG GEOparse - PLATFORM: GPL570
14-Jun-2026 19:31:07 DEBUG GEOparse - SAMPLE: GSM372286
14-Jun-2026 19:31

Download complete!


In [17]:
print("Number of samples:", len(gse2.gsms))

# print first 15 sample titles to see the labeling pattern
for name, sample in list(gse2.gsms.items())[15:35]:
    title = sample.metadata["title"][0]
    print(name, "→", title)

Number of samples: 82
GSM372301 → Normal Skin, Normal-16
GSM372302 → Normal Skin, Normal-17
GSM372303 → Normal Skin, Normal-18
GSM372304 → Normal Skin, Normal-19
GSM372305 → Normal Skin, Normal-20
GSM372306 → Normal Skin, Normal-21
GSM372307 → Uninvolved Skin, NS-1
GSM372308 → Lesion Skin, LS-1
GSM372309 → Uninvolved Skin, NS-2
GSM372310 → Lesion Skin, LS-2
GSM372311 → Uninvolved Skin, NS-3
GSM372312 → Lesion Skin, LS-3
GSM372313 → Uninvolved Skin, NS-4
GSM372314 → Lesion Skin, LS-4
GSM372315 → Uninvolved Skin, NS-5
GSM372316 → Lesion Skin, LS-5
GSM372317 → Uninvolved Skin, NS-6
GSM372318 → Lesion Skin, LS-6
GSM372319 → Uninvolved Skin, NS-7
GSM372320 → Lesion Skin, LS-7


In [9]:
import re

# extract the suffix after the underscore for every sample
suffixes = set()
for name, sample in gse2.gsms.items():
    title = sample.metadata["title"][0]
    suffix = title.split("_")[-1]
    suffixes.add(suffix)

print("Unique suffixes found:", suffixes)

Unique suffixes found: {'NL', 'LS'}


mwahahaha its the EXACT SAME :D

In [18]:
val_samples = {}
val_labels = {}
for name, sample in gse2.gsms.items():
    title = sample.metadata["title"][0]
    if title.startswith("Lesion Skin"):
        val_samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        val_labels[name] = 1  # psoriasis
    elif title.startswith("Normal Skin"):
        val_samples[name] = sample.table.set_index("ID_REF")["VALUE"]
        val_labels[name] = 0  # healthy
df_val = pd.DataFrame(val_samples).T
y_val = pd.Series(val_labels)

print("Validation samples:", df_val.shape[0])
print("Genes per sample:", df_val.shape[1])
print("Psoriasis (1):", sum(y_val == 1))
print("Healthy (0):", sum(y_val == 0))

Validation samples: 54
Genes per sample: 54675
Psoriasis (1): 33
Healthy (0): 21


In [19]:
#checking for any overlaps between this dataset and our og one
genes_found = [g for g in top_genes if g in df_val.columns]
print("Genes found:", len(genes_found), "out of", len(top_genes))

Genes found: 300 out of 300


In [22]:
df_val_filtered = df_val[top_genes]

# scale using the SAME scaler from training (we do NOT refit it!)
X_val_scaled = scaler.transform(df_val_filtered)

# get predictions using our saved model (no retraining!)
y_val_pred = model.predict(X_val_scaled)
y_val_prob = model.predict_proba(X_val_scaled)[:, 1]

# evaluate
from sklearn.metrics import classification_report, roc_auc_score
print("Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=['Healthy', 'Psoriasis']))
print("AUROC:", round(roc_auc_score(y_val, y_val_prob), 4))

=== External Validation Results (GSE14905) ===

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.00      0.00      0.00        21
   Psoriasis       0.61      1.00      0.76        33

    accuracy                           0.61        54
   macro avg       0.31      0.50      0.38        54
weighted avg       0.37      0.61      0.46        54

AUROC: 0.5


C:\Users\artkm\Documents\ai-psoriasis-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\artkm\Documents\ai-psoriasis-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\artkm\Documents\ai-psoriasis-project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

In [24]:
# reload our filtered training data
df_filtered = pd.read_csv("data/filtered_expression.csv", index_col=0)
print("Loaded! Shape:", df_filtered.shape)
# pick one of our top genes and compare its average value across the two datasets
gene = top_genes[0]

print(f"Gene: {gene}")
print("Average value in TRAINING data (GSE13355):", df_filtered[gene].mean().round(3))
print("Average value in VALIDATION data (GSE14905):", df_val_filtered[gene].mean().round(3))

Loaded! Shape: (122, 300)
Gene: 203691_at
Average value in TRAINING data (GSE13355): 0.86
Average value in VALIDATION data (GSE14905): 10.503


oh shit D:

In [26]:
print("Average across ALL 300 genes:")
print("Training data (GSE13355):", df_filtered[top_genes].mean().mean().round(3))
print("Validation data (GSE14905):", df_val_filtered.mean().mean().round(3))

Average across ALL 300 genes:
Training data (GSE13355): 0.225
Validation data (GSE14905): 8.626


In [27]:
#oh god

In [29]:
from sklearn.preprocessing import StandardScaler

# create a NEW scaler that learns mean/std from THIS dataset's own values
val_scaler = StandardScaler()
X_val_scaled_v2 = val_scaler.fit_transform(df_val_filtered)

y_val_pred_v2 = model.predict(X_val_scaled_v2)
y_val_prob_v2 = model.predict_proba(X_val_scaled_v2)[:, 1]

print("Classification Report:")
print(classification_report(y_val, y_val_pred_v2, target_names=['Healthy', 'Psoriasis']))
print("AUROC:", round(roc_auc_score(y_val, y_val_prob_v2), 4)) 
#plsspsplsplspls

Classification Report:
              precision    recall  f1-score   support

     Healthy       0.91      0.95      0.93        21
   Psoriasis       0.97      0.94      0.95        33

    accuracy                           0.94        54
   macro avg       0.94      0.95      0.94        54
weighted avg       0.95      0.94      0.94        54

AUROC: 0.9957
